# SPY Hybrid QRNN SI Self-lagged

Original unrestricted model tuning with seven separately estimated quantile levels and rank-based model selection.

In [2]:
# ============================================================
# SPY MODELLING DATA PREPARATION
# Common setup for all ERNN and QRNN specifications
# ============================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Reproducibility and device
# ------------------------------------------------------------

BASE_SEED = 2026

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. Load the scaled SPY dataset
# ------------------------------------------------------------

DATA_PATH = Path.home() / "SPY_features_scaled.csv"

df = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=True
)

df = (
    df
    .sort_index()
    .drop_duplicates()
    .dropna()
    .copy()
)


# ------------------------------------------------------------
# 3. Response and predictors
# ------------------------------------------------------------

target_col = "log_return"

feature_cols = [
    "Volatility",
    "RSI_14",
    "ATR_14",
    "QQQ_LogReturns",
    "VIX_LogChange",
    "Volume_LogChange",
]

required_cols = [target_col] + feature_cols

missing_cols = [
    column
    for column in required_cols
    if column not in df.columns
]

if missing_cols:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_cols)
    )


# ------------------------------------------------------------
# 4. Chronological 70%-15%-15% split
# ------------------------------------------------------------

train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size:].copy()


# ------------------------------------------------------------
# 5. Convert samples to tensors
# ------------------------------------------------------------

def dataframe_to_tensors(sample):
    X = torch.tensor(
        sample[feature_cols].to_numpy(),
        dtype=torch.float32,
        device=device
    )
    y = torch.tensor(
        sample[target_col].to_numpy(),
        dtype=torch.float32,
        device=device
    )
    return X, y


X_train, y_train = dataframe_to_tensors(train_df)
X_val, y_val = dataframe_to_tensors(val_df)
X_test, y_test = dataframe_to_tensors(test_df)

# Compatibility aliases used by later QRNN cells.
X_test_q = X_test
y_test_q = y_test

levels = [
    0.025,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.975,
]

alpha_levels = levels
tau_levels = levels


# ------------------------------------------------------------
# 6. Checks and split summary
# ------------------------------------------------------------

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
assert X_train.shape[1] == len(feature_cols)
assert torch.isfinite(X_train).all()
assert torch.isfinite(X_val).all()
assert torch.isfinite(X_test).all()
assert torch.isfinite(y_train).all()
assert torch.isfinite(y_val).all()
assert torch.isfinite(y_test).all()

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Observations": [len(train_df), len(val_df), len(test_df)],
    "Start": [
        train_df.index.min().date(),
        val_df.index.min().date(),
        test_df.index.min().date(),
    ],
    "End": [
        train_df.index.max().date(),
        val_df.index.max().date(),
        test_df.index.max().date(),
    ],
})

display(split_summary)

print("Input features:", feature_cols)
print("Number of predictors:", len(feature_cols))
print("X_train shape:", tuple(X_train.shape))
print("X_val shape:  ", tuple(X_val.shape))
print("X_test shape: ", tuple(X_test.shape))


Device: cpu


,Sample,Observations,Start,End
0,Training,2219,2014-01-23,2022-11-11
1,Validation,475,2022-11-14,2024-10-04
2,Test,476,2024-10-07,2026-08-31


Input features: ['Volatility', 'RSI_14', 'ATR_14', 'QQQ_LogReturns', 'VIX_LogChange', 'Volume_LogChange']
Number of predictors: 6
X_train shape: (2219, 6)
X_val shape:   (475, 6)
X_test shape:  (476, 6)


## Original tuning

Run after the preparation cell.

In [ ]:
# ============================================================
# ORIGINAL TUNING — SPY HYBRID QRNN SI SELF-LAGGED
# Seven separately estimated levels; 200 Optuna trials per level
# ============================================================

import copy
import json
import time

import optuna
import torch.nn as nn
import torch.nn.functional as F
from optuna.samplers import TPESampler


MODEL_NAME = "spy_qrnn_si_self"
FORM = "SI"
LAG_TYPE = "self"
N_TRIALS = 200
MAX_EPOCHS = 200

ALPHA_LEVELS = [
    0.025, 0.050, 0.250, 0.500,
    0.750, 0.950, 0.975,
]

OUTPUT_DIR = Path("spy_original_tuning_results") / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR.resolve())


def set_model_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def hybrid_qrnn_loss(y, q_hybrid, sigma_al, alpha):
    """Mean asymmetric-Laplace negative log-likelihood."""
    eps = torch.finfo(q_hybrid.dtype).eps
    sigma_al = torch.clamp(sigma_al, min=eps)
    alpha_tensor = torch.as_tensor(
        alpha,
        dtype=q_hybrid.dtype,
        device=q_hybrid.device
    )
    residual = y - q_hybrid
    indicator = (residual < 0).to(q_hybrid.dtype)
    check_loss = residual * (alpha_tensor - indicator)
    nll = (
        -torch.log(alpha_tensor * (1.0 - alpha_tensor))
        + torch.log(sigma_al)
        + check_loss / sigma_al
    )
    return nll.mean()


def compute_hybrid_location(y, X, beta1, beta2, beta3, fnn, psi):
    """SI-form self-lagged CAViaR-hybrid recursion."""
    fnn_output = fnn(X).squeeze(-1)
    zero_value = y.new_zeros(())
    statistical_forecasts = [zero_value]
    hybrid_forecasts = [zero_value]

    for time_index in range(1, len(y)):
        previous_return = y[time_index - 1]
        previous_state = statistical_forecasts[time_index - 1]
        absolute_return = torch.abs(previous_return)
        statistical_value = (
            beta1 * previous_state
            + beta2 * absolute_return
            + beta3 * previous_state * absolute_return
        )
        hybrid_value = (
            psi * statistical_value
            + (1.0 - psi) * fnn_output[time_index]
        )
        statistical_forecasts.append(statistical_value)
        hybrid_forecasts.append(hybrid_value)

    return (
        torch.stack(hybrid_forecasts),
        torch.stack(statistical_forecasts),
    )


class HybridQRNN_SI_Self(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        self.eps = 1e-6
        self.beta1_raw = nn.Parameter(torch.tensor(1.386))
        self.beta2_raw = nn.Parameter(torch.tensor(0.3))
        self.beta3_raw = nn.Parameter(
            torch.tensor(0.3)
        )
        self.psi_raw = nn.Parameter(torch.tensor(0.0))
        self.sigma_raw = nn.Parameter(torch.tensor(0.5))

        layers = []
        previous_dimension = input_dim
        for layer_index in range(config["n_hidden_layers"]):
            hidden_dimension = config[f"hidden_dim_{layer_index}"]
            layers.append(nn.Linear(previous_dimension, hidden_dimension))
            if config["layernorm"]:
                layers.append(nn.LayerNorm(hidden_dimension))
            layers.append(self.get_activation(config["activation"]))
            if config["dropout"] > 0:
                layers.append(nn.Dropout(config["dropout"]))
            previous_dimension = hidden_dimension
        layers.append(nn.Linear(previous_dimension, 1))
        self.fnn = nn.Sequential(*layers)
        self.initialise_weights(config["weight_init"])

    @staticmethod
    def get_activation(name):
        return {
            "relu": nn.ReLU(),
            "elu": nn.ELU(),
            "gelu": nn.GELU(),
            "tanh": nn.Tanh(),
        }[name]

    def initialise_weights(self, method):
        for module in self.fnn.modules():
            if not isinstance(module, nn.Linear):
                continue
            if method == "glorot":
                nn.init.xavier_uniform_(module.weight)
            else:
                nn.init.kaiming_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, y, X):
        beta1 = torch.sigmoid(self.beta1_raw)
        beta2 = F.softplus(self.beta2_raw)
        beta3 = F.softplus(
            self.beta3_raw
        )
        psi = torch.sigmoid(self.psi_raw)
        sigma = F.softplus(self.sigma_raw) + self.eps
        hybrid, statistical = compute_hybrid_location(
            y, X, beta1, beta2, beta3, self.fnn, psi
        )
        return hybrid, sigma, statistical


def calculate_crp(y, forecast, alpha):
    if alpha <= 0.5:
        empirical = (y <= forecast).float().mean().item()
        target = alpha
    else:
        empirical = (y > forecast).float().mean().item()
        target = 1.0 - alpha
    crp = empirical / target
    return crp, (crp - 1.0) ** 2, empirical


def suggest_config(trial):
    config = {
        "n_hidden_layers": trial.suggest_int("n_hidden_layers", 1, 3),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "elu", "gelu", "tanh"]
        ),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "layernorm": trial.suggest_categorical("layernorm", [True, False]),
        "weight_init": trial.suggest_categorical(
            "weight_init", ["glorot", "he"]
        ),
        "lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
        "optimizer": trial.suggest_categorical(
            "optimizer", ["adam", "adamw"]
        ),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 1e-4),
        "gradient_clip": trial.suggest_categorical(
            "gradient_clip", [0.1, 1.0, 5.0]
        ),
        "patience": trial.suggest_categorical(
            "patience", [10, 20, 30]
        ),
    }
    for layer_index in range(config["n_hidden_layers"]):
        config[f"hidden_dim_{layer_index}"] = trial.suggest_int(
            f"hidden_dim_{layer_index}", 32, 256
        )
    return config


def make_optimizer(model, config):
    arguments = {
        "params": model.parameters(),
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
    }
    if config["optimizer"] == "adam":
        return torch.optim.Adam(**arguments)
    return torch.optim.AdamW(**arguments)


def train_configuration(config, alpha, model_seed=BASE_SEED):
    set_model_seed(model_seed)
    model = HybridQRNN_SI_Self(X_train.shape[1], config).to(device)
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=10,
        factor=0.5,
        min_lr=1e-6,
    )
    best_nll = np.inf
    best_state = None
    patience_counter = 0
    epochs_completed = 0

    for epoch in range(MAX_EPOCHS):
        epochs_completed = epoch + 1
        model.train()
        optimizer.zero_grad()
        train_forecast, train_sigma, _ = model(y_train, X_train)
        train_loss = hybrid_qrnn_loss(
            y_train, train_forecast, train_sigma, alpha
        )
        if not torch.isfinite(train_loss):
            return None
        train_loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(), config["gradient_clip"]
        )
        optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_forecast, validation_sigma, _ = model(y_val, X_val)
            validation_loss = hybrid_qrnn_loss(
                y_val, validation_forecast, validation_sigma, alpha
            )
        if not torch.isfinite(validation_loss):
            return None
        scheduler.step(validation_loss)
        current_nll = validation_loss.item()
        if current_nll < best_nll:
            best_nll = current_nll
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config["patience"]:
                break

    if best_state is None:
        return None
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        validation_forecast, validation_sigma, _ = model(y_val, X_val)
        validation_nll = hybrid_qrnn_loss(
            y_val, validation_forecast, validation_sigma, alpha
        ).item()
        crp, crp_penalty, hit_rate = calculate_crp(
            y_val, validation_forecast, alpha
        )
    parameters = {
        "beta1": torch.sigmoid(model.beta1_raw).item(),
        "beta2": F.softplus(model.beta2_raw).item(),
        "beta3": (
            F.softplus(model.beta3_raw).item()
            if model.beta3_raw is not None
            else np.nan
        ),
        "psi": torch.sigmoid(model.psi_raw).item(),
        "sigma": (F.softplus(model.sigma_raw) + model.eps).item(),
    }
    return {
        "model": model,
        "validation_nll": validation_nll,
        "validation_crp": crp,
        "validation_crp_penalty": crp_penalty,
        "validation_hit_rate": hit_rate,
        "epochs_completed": epochs_completed,
        **parameters,
    }


def objective(trial, alpha):
    config = suggest_config(trial)
    result = train_configuration(config, alpha)
    if result is None:
        raise optuna.TrialPruned("Non-finite loss or no valid checkpoint.")
    for name in [
        "validation_nll", "validation_crp", "validation_crp_penalty",
        "validation_hit_rate", "epochs_completed", "beta1", "beta2",
        "beta3", "psi", "sigma",
    ]:
        trial.set_user_attr(name, result[name])
    return result["validation_nll"]


def rank_trials(study):
    trials = [
        trial for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and trial.value is not None
        and np.isfinite(trial.value)
        and "validation_crp_penalty" in trial.user_attrs
    ]
    if not trials:
        raise RuntimeError("No valid completed Optuna trials.")
    nll_order = sorted(
        trials, key=lambda trial: trial.user_attrs["validation_nll"]
    )
    crp_order = sorted(
        trials,
        key=lambda trial: trial.user_attrs["validation_crp_penalty"],
    )
    nll_rank = {trial.number: rank for rank, trial in enumerate(nll_order, 1)}
    crp_rank = {trial.number: rank for rank, trial in enumerate(crp_order, 1)}
    records = []
    for trial in trials:
        record = {
            "trial": trial.number,
            **trial.user_attrs,
            "rank_nll": nll_rank[trial.number],
            "rank_crp": crp_rank[trial.number],
            "sum_rank": nll_rank[trial.number] + crp_rank[trial.number],
            **trial.params,
        }
        records.append(record)
    table = pd.DataFrame(records).sort_values(
        [
            "sum_rank",
            "validation_nll",
            "validation_crp_penalty",
            "trial",
        ]
    ).reset_index(drop=True)
    selected_number = int(table.iloc[0]["trial"])
    selected_trial = next(
        trial for trial in trials if trial.number == selected_number
    )
    return selected_trial, table


best_configs_spy_qrnn_si_self = {}
best_models_spy_qrnn_si_self = {}
summary_records = []
overall_start = time.time()

for level_index, alpha in enumerate(ALPHA_LEVELS):
    level_start = time.time()
    sampler_seed = BASE_SEED + level_index
    print("\n" + "=" * 72)
    print(f"Tuning {MODEL_NAME} at alpha={alpha:.3f}")
    print(f"Trials={N_TRIALS} | sampler seed={sampler_seed}")
    print("=" * 72)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=sampler_seed),
        study_name=f"{MODEL_NAME}_alpha_{alpha:.3f}",
    )
    study.optimize(
        lambda trial, current_alpha=alpha: objective(trial, current_alpha),
        n_trials=N_TRIALS,
        show_progress_bar=True,
        gc_after_trial=True,
    )
    selected_trial, ranking_table = rank_trials(study)
    alpha_label = f"{alpha:.3f}"
    ranking_table.insert(0, "alpha", alpha)
    ranking_path = OUTPUT_DIR / (
        f"{MODEL_NAME}_ranking_alpha_{alpha_label}.csv"
    )
    ranking_table.to_csv(ranking_path, index=False)

    selected_config = selected_trial.params.copy()
    selected_result = train_configuration(selected_config, alpha)
    if selected_result is None:
        raise RuntimeError(
            f"Selected trial could not be reproduced for alpha={alpha}."
        )
    selected_model = selected_result["model"]
    checkpoint_path = OUTPUT_DIR / (
        f"{MODEL_NAME}_best_alpha_{alpha_label}.pt"
    )
    torch.save(
        {
            "model_name": MODEL_NAME,
            "asset": "SPY",
            "form": FORM,
            "lag_type": LAG_TYPE,
            "alpha": alpha,
            "selected_trial": selected_trial.number,
            "config": selected_config,
            "model_state_dict": {
                key: value.detach().cpu()
                for key, value in selected_model.state_dict().items()
            },
            "validation_nll": selected_result["validation_nll"],
            "validation_crp": selected_result["validation_crp"],
            "validation_crp_penalty": selected_result[
                "validation_crp_penalty"
            ],
            "validation_hit_rate": selected_result["validation_hit_rate"],
            "estimated_parameters": {
                name: selected_result[name]
                for name in ["beta1", "beta2", "beta3", "psi", "sigma"]
            },
            "feature_cols": feature_cols,
            "number_of_trials": N_TRIALS,
            "base_seed": BASE_SEED,
            "sampler_seed": sampler_seed,
        },
        checkpoint_path,
    )
    selected_record = {
        **selected_config,
        "selected_trial": selected_trial.number,
        **{
            name: selected_result[name]
            for name in [
                "validation_nll", "validation_crp",
                "validation_crp_penalty", "validation_hit_rate",
                "beta1", "beta2", "beta3", "psi", "sigma",
            ]
        },
    }
    best_configs_spy_qrnn_si_self[alpha] = selected_record
    best_models_spy_qrnn_si_self[alpha] = selected_model
    runtime_hours = (time.time() - level_start) / 3600
    summary_records.append({
        "alpha": alpha,
        "selected_trial": selected_trial.number,
        "validation_nll": selected_result["validation_nll"],
        "validation_crp": selected_result["validation_crp"],
        "validation_crp_penalty": selected_result[
            "validation_crp_penalty"
        ],
        "validation_hit_rate": selected_result["validation_hit_rate"],
        "beta1": selected_result["beta1"],
        "beta2": selected_result["beta2"],
        "beta3": selected_result["beta3"],
        "psi": selected_result["psi"],
        "sigma": selected_result["sigma"],
        "valid_trials": len(ranking_table),
        "runtime_hours": runtime_hours,
    })
    print("\nTop 10 rank-based trials:")
    print(ranking_table[[
        "trial", "validation_nll", "validation_crp",
        "validation_crp_penalty", "rank_nll", "rank_crp", "sum_rank",
    ]].head(10).to_string(index=False))
    print("\nSelected trial:", selected_trial.number)
    print("Validation NLL:", f"{selected_result['validation_nll']:.8f}")
    print("Validation CRP:", f"{selected_result['validation_crp']:.8f}")
    print("Ranking saved to:", ranking_path)
    print("Checkpoint saved to:", checkpoint_path)


summary_spy_qrnn_si_self = pd.DataFrame(summary_records)
summary_path = OUTPUT_DIR / f"{MODEL_NAME}_tuning_summary.csv"
summary_spy_qrnn_si_self.to_csv(summary_path, index=False)

config_path = OUTPUT_DIR / f"{MODEL_NAME}_selected_configs.json"
with open(config_path, "w") as configuration_file:
    json.dump(
        {f"{alpha:.3f}": config for alpha, config in best_configs_spy_qrnn_si_self.items()},
        configuration_file,
        indent=2,
    )

print("\n" + "=" * 72)
print("SPY ORIGINAL TUNING COMPLETED:", MODEL_NAME)
print("=" * 72)
display(summary_spy_qrnn_si_self)
print("Total runtime (hours):", f"{(time.time() - overall_start) / 3600:.3f}")
print("Summary saved to:", summary_path)
print("Selected configurations saved to:", config_path)


[I 2026-09-14 14:12:53,869] A new study created in memory with name: spy_qrnn_si_self_alpha_0.025


Model: spy_qrnn_si_self
Output directory: /Users/miaomiaochen/Desktop/PhD program/Project 1/spy_original_tuning_results/spy_qrnn_si_self

Tuning spy_qrnn_si_self at alpha=0.025
Trials=200 | sampler seed=2026


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-09-14 14:13:19,063] Trial 0 finished with value: 3.619342565536499 and parameters: {'n_hidden_layers': 1, 'activation': 'elu', 'dropout': 0.49377524711883497, 'layernorm': False, 'weight_init': 'he', 'lr': 0.0006552020421842752, 'optimizer': 'adam', 'weight_decay': 8.940884610144999e-05, 'gradient_clip': 5.0, 'patience': 30, 'hidden_dim_0': 38}. Best is trial 0 with value: 3.619342565536499.
[I 2026-09-14 14:13:50,539] Trial 1 finished with value: 2.566314458847046 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.07458356653667625, 'layernorm': False, 'weight_init': 'he', 'lr': 0.006969178688110181, 'optimizer': 'adam', 'weight_decay': 2.574006001095316e-05, 'gradient_clip': 1.0, 'patience': 30, 'hidden_dim_0': 102}. Best is trial 1 with value: 2.566314458847046.
[I 2026-09-14 14:13:53,588] Trial 2 finished with value: 3.6926815509796143 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.37112272198735796, 'layernorm': True, 'weig

[I 2026-09-14 15:36:47,286] A new study created in memory with name: spy_qrnn_si_self_alpha_0.050



Top 10 rank-based trials:
 trial  validation_nll  validation_crp  validation_crp_penalty  rank_nll  rank_crp  sum_rank
   186       -3.735312        1.010526                0.000111         8         5        13
    82       -3.748983        0.926316                0.005429         5        10        15
    72       -3.710183        1.010526                0.000111        14         2        16
    71       -3.703863        1.094737                0.008975        17        15        32
    51       -3.680652        0.926316                0.005429        27         8        35
    43       -3.774155        0.757895                0.058615         2        36        38
    12       -3.665908        0.926316                0.005429        31         7        38
   179       -3.718022        0.842105                0.024931        12        28        40
   185       -3.658540        0.926316                0.005429        33        11        44
   188       -3.699901        1.094737     

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-09-14 15:37:15,798] Trial 0 finished with value: 2.887434720993042 and parameters: {'n_hidden_layers': 3, 'activation': 'relu', 'dropout': 0.05498225107357002, 'layernorm': True, 'weight_init': 'he', 'lr': 0.001007193483885585, 'optimizer': 'adam', 'weight_decay': 5.273500806799822e-05, 'gradient_clip': 5.0, 'patience': 20, 'hidden_dim_0': 146, 'hidden_dim_1': 251, 'hidden_dim_2': 82}. Best is trial 0 with value: 2.887434720993042.
[I 2026-09-14 15:37:39,980] Trial 1 finished with value: 3.0044784545898438 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.025049491897366605, 'layernorm': True, 'weight_init': 'glorot', 'lr': 0.00015333527280501685, 'optimizer': 'adam', 'weight_decay': 1.7649100996182442e-05, 'gradient_clip': 0.1, 'patience': 30, 'hidden_dim_0': 55}. Best is trial 0 with value: 2.887434720993042.
[I 2026-09-14 15:38:06,854] Trial 2 finished with value: -0.28163060545921326 and parameters: {'n_hidden_layers': 2, 'activation': 'elu', 'dropou